# Random Forest

In [1]:
import pandas as pd
import numpy as np
import math
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score, make_scorer, roc_auc_score, accuracy_score
from pathlib import Path
import time
from datetime import timedelta
import re

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare training
# Lista dei csv su cui fare training
# datasets = {
#     'ambl_lesions_radiomic' : FILE_PATH / 'ambl_lesions_radiomic_medsam.csv',
#     'duke_lesions_radiomic' : FILE_PATH / 'duke_lesions_radiomic_medsam.csv',
#     'ambl_lesions' : FILE_PATH / 'ambl_lesions.csv',
#     'duke_lesions' : FILE_PATH / 'duke_lesions.csv',
#     't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
#     't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
#     't2_original': FILE_PATH / 't2_original_masks.csv',
#     'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
#     'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
#     'original_dynamic': FILE_PATH / 'original_dynamic.csv'
# }

datasets = {
    'duke_lesions_radiomic' : FILE_PATH / 'duke_lesions_radiomic_medsam.csv',
    'duke_lesions' : FILE_PATH / 'duke_lesions.csv',
    'ambl_lesions_radiomic' : FILE_PATH / 'ambl_lesions_radiomic_medsam.csv',
    'ambl_lesions' : FILE_PATH / 'ambl_lesions.csv'
}


# datasets = {
#     'duke_lesions_radiomic' : FILE_PATH / 'duke_lesions_radiomic_medsam.csv',
#     'duke_lesions' : FILE_PATH / 'duke_lesions.csv'
# }"""


# datasets = {
#     'ambl_lesions_radiomic' : FILE_PATH / 'ambl_lesions_radiomic_medsam.csv',
#     'ambl_lesions' : FILE_PATH / 'ambl_lesions.csv'
# }

In [2]:
def best_threshold(y_true, y_prob):
    thresholds = np.linspace(0.05, 0.95, 50)
    scores = [
        f1_score(
            y_true,
            (y_prob >= t).astype(int),
            average="macro",
            zero_division=0
        )
        for t in thresholds
    ]
    return thresholds[np.argmax(scores)]


# Training Duke

In [3]:
def training_duke(file_path, csv_name):
    import pandas as pd
    import numpy as np
    import re
    from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.multioutput import MultiOutputClassifier
    from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

    df = pd.read_csv(file_path)

    if "Patient ID" not in df.columns:
        raise ValueError(f"{csv_name} - manca 'Patient ID'")

    # ================= TARGET =================
    df["ER_class"]   = pd.to_numeric(df["ER"], errors="coerce")
    df["PR_class"]   = pd.to_numeric(df["PR"], errors="coerce")
    df["HER2_class"] = pd.to_numeric(df["HER2"], errors="coerce")

    final_target_list = ["ER_class", "PR_class", "HER2_class"]
    df = df.dropna(subset=final_target_list).copy()
    for c in final_target_list:
        df[c] = df[c].astype(int)

    final_target_list = [c for c in final_target_list if df[c].nunique() > 1]
    if len(final_target_list) == 0:
        return None

    # ================= FEATURES =================
    features_to_drop = [
        "Patient ID", "lesion idx", "tumor/benign",
        "GRADE", "isTN", "Breast",
        "ER", "PR", "HER2"
    ] + final_target_list

    X = df.drop(columns=features_to_drop, errors="ignore")
    y = df[final_target_list]
    groups = df["Patient ID"]

    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.mean(numeric_only=True))
    X.columns = [re.sub(r"\[|\]|<", "", c) for c in X.columns]

    # ================= CV =================
    y_strat = y["HER2_class"].astype(str)
    n_splits = min(5, y_strat.value_counts().min())
    if n_splits < 2:
        return None

    sgkf = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )
    splits = list(sgkf.split(X, y_strat, groups))

    # ================= GRID SEARCH =================
    base_model = RandomForestClassifier(
        random_state=42,
        n_jobs=1,
        class_weight="balanced_subsample"
    )

    def multi_f1_scorer(y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        return np.mean([
            f1_score(
                y_true[:, i],
                y_pred[:, i],
                average="macro",
                zero_division=0
            )
            for i in range(y_true.shape[1])
        ])

    scorer = make_scorer(multi_f1_scorer)

    grid = GridSearchCV(
        MultiOutputClassifier(base_model),
        param_grid={
            "estimator__n_estimators": [50, 75, 100],
            "estimator__max_depth": [2, 4, 6],
            "estimator__min_samples_leaf": [1, 2, 3]
        },
        scoring=scorer,
        cv=splits,
        n_jobs=-1,
        error_score="raise"
    )

    grid.fit(X, y)

    best_params = {
        k.replace("estimator__", ""): v
        for k, v in grid.best_params_.items()
    }

    # ================= CV EVALUATION =================
    fold_reports = []

    for tr, te in splits:
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]

        model = MultiOutputClassifier(
            RandomForestClassifier(
                **best_params,
                random_state=42,
                n_jobs=1,
                class_weight="balanced_subsample"
            )
        )

        model.fit(X_tr, y_tr)
        y_proba_list = model.predict_proba(X_te)

        fold_metrics = {}

        for i, col in enumerate(final_target_list):
            proba_tr = model.predict_proba(X_tr)[i][:, 1]
            proba_te = y_proba_list[i][:, 1]

            t_opt = best_threshold(y_tr.iloc[:, i].values, proba_tr)
            y_pred = (proba_te >= t_opt).astype(int)

            fold_metrics[col] = {
                "f1": f1_score(y_te.iloc[:, i], y_pred, zero_division=0),
                "accuracy": accuracy_score(y_te.iloc[:, i], y_pred),
                "auc": roc_auc_score(y_te.iloc[:, i], proba_te)
            }

        fold_reports.append(fold_metrics)

    return {
        "best_params": best_params,
        "mean_score": grid.best_score_,
        "std_score": grid.cv_results_["std_test_score"][grid.best_index_],
        "fold_reports": fold_reports,
        "targets_used": final_target_list,
        "cv_results": grid.cv_results_
    }


# Training ambl

In [4]:
def training_ambl(file_path, csv_name):
    df = pd.read_csv(file_path)

    if "Patient ID" not in df.columns:
        raise ValueError(f"{csv_name} - manca 'Patient ID' necessario per Group split")

    df_validi = df.copy()

    # --- Controllo colonne target attese ---
    required_targets = ["ER [SII]", "PR [SII]", "HER2 [SII]"]
    missing = [c for c in required_targets if c not in df_validi.columns]
    if missing:
        print(f"[ERRORE] {csv_name} - mancano colonne target: {missing}")
        return None

    # --- Creo i target binari direttamente (già binari nel Duke) ---
    final_target_list = ["ER_class", "PR_class", "HER2_class"]

    df_validi["ER_class"]   = pd.to_numeric(df_validi["ER [SII]"], errors="coerce")
    df_validi["PR_class"]   = pd.to_numeric(df_validi["PR [SII]"], errors="coerce")
    df_validi["HER2_class"] = pd.to_numeric(df_validi["HER2 [SII]"], errors="coerce")

    # Tengo solo righe con tutti i target presenti e casto a int
    df_validi = df_validi.dropna(subset=final_target_list).copy()
    for col in final_target_list:
        df_validi[col] = df_validi[col].astype(int)

    # --- Tolgo target con 1 sola classe ---
    targets_da_rimuovere = []
    for col in final_target_list:
        if df_validi[col].nunique() < 2:
            print(f"[ATTENZIONE] {csv_name} - Target {col} ha una sola classe. Lo escludo.")
            targets_da_rimuovere.append(col)

    for col in targets_da_rimuovere:
        final_target_list.remove(col)

    if len(final_target_list) == 0:
        print(f"[ERRORE] {csv_name} - Nessun target valido (>=2 classi).")
        return None

    # --- Features / Target / Groups ---
    raw_target_cols = ["ER", "PR", "HER2"]

    features_to_drop = [
        "Patient ID", "lesion idx", "tumor/benign", "GRADE", "isTN", "Breast"
    ] + raw_target_cols + final_target_list

    features = df_validi.drop(columns=features_to_drop, errors="ignore")
    target = df_validi[final_target_list]
    groups = df_validi["Patient ID"]

    # Imputazione features numeriche
    features = features.apply(pd.to_numeric, errors="coerce")
    features = features.fillna(features.mean(numeric_only=True))

    # Pulizia nomi colonne
    features.columns = [re.sub(r"\[|\]|<", "", col) for col in features.columns]

    # --- StratifiedGroupKFold ---
    if "HER2_class" in final_target_list:
        y_strat = target["HER2_class"].astype(str)
        n_pos = int(target["HER2_class"].sum())
        n_splits = 3 if n_pos < 10 else 5
    else:
        y_strat = target.astype(int).astype(str).agg("_".join, axis=1)
        n_splits = 5


    # Debug
    min_class_count = y_strat.value_counts().min()
    n_splits = min(n_splits, min_class_count)

    if n_splits < 2:
        print(f"[ERRORE] {csv_name} - troppo pochi campioni per CV stratificata")
        return None


    vc = y_strat.value_counts()
    rare = vc[vc < n_splits].index
    if len(rare) > 0:
        y_strat = y_strat.where(~y_strat.isin(rare), other="RARE")

    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
    splits = list(sgkf.split(features, y_strat, groups=groups))

    # Debug utilissimo: controlla positivi HER2 per fold
    """if "HER2_class" in final_target_list:
        for k, (tr, te) in enumerate(splits):
            tr_pos = int(target.iloc[tr]["HER2_class"].sum())
            te_pos = int(target.iloc[te]["HER2_class"].sum())
            print(f"[{csv_name}] Fold {k}: HER2 train pos={tr_pos} | test pos={te_pos}")
"""

    # Modello base Random Forest con parametri NON ottimizzati fissati
    base_model = RandomForestClassifier(
        random_state=42,
        n_jobs=1,
        class_weight='balanced_subsample',
        max_features='sqrt',   
        min_samples_split=2,   
        criterion='gini'       
    )

    multi_output_model = MultiOutputClassifier(base_model)

    iperparametri = {
        'estimator__n_estimators': [50, 75, 100],
        'estimator__max_depth': [2, 4, 6],
        'estimator__min_samples_leaf': [1, 2, 3]
    }

    # scorer: media della F1 macro sui target
    def multi_f1_scorer(y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        scores = []
        for i in range(y_true.shape[1]):
            scores.append(f1_score(y_true[:, i], y_pred[:, i], average='macro', zero_division=0))
        return float(np.mean(scores))

    scorer = make_scorer(multi_f1_scorer)

    total_combinations = math.prod(len(v) for v in iperparametri.values())
    #print(f"\nInizio Grid Search RF (GRID MINIMAL: {total_combinations} combinazioni) per: {csv_name}")

    grid_search = GridSearchCV(
        estimator=multi_output_model,
        param_grid=iperparametri,
        cv=splits,          
        scoring=scorer,
        n_jobs=-1,
        verbose=1,
        refit=True,
        return_train_score=False,
        error_score='raise'
    )

    grid_search.fit(features, target)

    best_params = grid_search.best_params_
    best_score  = grid_search.best_score_

    # tolgo il prefisso 'estimator__'
    clean_best_params = {k.replace('estimator__', ''): v for k, v in best_params.items()}


    # parametri finali del RF (fissi + ottimizzati)
    final_model_params = {
        'random_state': 42,
        'n_jobs': 1,
        'class_weight': 'balanced_subsample',
        'max_features': 'sqrt',
        'min_samples_split': 2,
        'criterion': 'gini',
        **clean_best_params
    }

    # Metriche per FOLD e per LABEL
    fold_reports = []

    for k, (train_idx, test_idx) in enumerate(splits):
        X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
        y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

        model_clone = MultiOutputClassifier(RandomForestClassifier(**final_model_params))
        model_clone.fit(X_train, y_train)

        y_pred = model_clone.predict(X_test)
        y_proba_list = model_clone.predict_proba(X_test)

         # DEBUG: Fold k
        print(f"\n[DEBUG] Fold {k}")
        for i, col in enumerate(final_target_list):
            proba_i = y_proba_list[i]
            y_true_i = y_test.iloc[:, i].values

            print(
                f"  Target: {col} | "
                f"y_true classes: {np.unique(y_true_i)} | "
                f"proba shape: {proba_i.shape}"
            )



        fold_metrics = {}
        for i, col in enumerate(final_target_list):
            y_true_i = y_test.iloc[:, i]
            y_pred_i = y_pred[:, i]

            f1 = f1_score(y_true_i, y_pred_i, average="macro", zero_division=0)
            acc = accuracy_score(y_true_i, y_pred_i)

            auc_val = np.nan
            proba_i = y_proba_list[i]

            # Caso BINARIO
            if len(np.unique(y_true_i)) == 2 and proba_i.shape[1] == 2:
                auc_val = roc_auc_score(y_true_i, proba_i[:, 1])

            # Caso MULTICLASSE (AMBL)
            elif len(np.unique(y_true_i)) > 2:
                try:
                    auc_val = roc_auc_score(
                        y_true_i,
                        proba_i,
                        multi_class="ovr",
                        average="macro"
                    )
                except ValueError:
                    auc_val = np.nan

            fold_metrics[col] = {
                'f1': f1,
                'accuracy': acc,
                'auc': auc_val
            }
        # Debug
        print(f"\nMetriche Fold {k}")
        for col, m in fold_metrics.items():
            auc_str = "nan" if np.isnan(m["auc"]) else f"{m['auc']:.3f}"
            print(
                f"  {col}: "
                f"F1={m['f1']:.3f} | "
                f"ACC={m['accuracy']:.3f} | "
                f"AUC={auc_str}"
            )

        fold_reports.append(fold_metrics)

    final_result = {
        **clean_best_params,
        'mean_score': best_score,
        'std_score': grid_search.cv_results_['std_test_score'][grid_search.best_index_],
        'fold_reports': fold_reports,
        'targets_used': final_target_list,
        "cv_results": grid_search.cv_results_ 
    }
    return final_result

# Stampo i risultati in un formato piú leggibile

In [5]:
import numpy as np
import pandas as pd
from pathlib import Path


def print_grid_search_results(results_per_dataset, save_csv=True, output_path="RandomForest.csv"):
    print("\n" + "=" * 80)
    print(" " * 20 + "Metriche (MEDIA ± STD) per target")
    print("=" * 80)

    # Per il csv
    rows = []

    for dataset_name, best_result in results_per_dataset.items():
        if best_result is None:
            continue

        fold_reports = best_result["fold_reports"]
        target_names = best_result.get("targets_used", [])

        print(f"\n\nDataset: {dataset_name}")
        print("-" * 80)

        for target_name in target_names:
            f1_list  = np.array([fold[target_name]["f1"] for fold in fold_reports], dtype=float)
            acc_list = np.array([fold[target_name]["accuracy"] for fold in fold_reports], dtype=float)
            auc_list = np.array([fold[target_name]["auc"] for fold in fold_reports], dtype=float)

            f1_mean,  f1_std  = np.mean(f1_list),  np.std(f1_list)
            acc_mean, acc_std = np.mean(acc_list), np.std(acc_list)

            valid_auc = ~np.isnan(auc_list)
            auc_mean = np.mean(auc_list[valid_auc]) if valid_auc.any() else np.nan
            auc_std  = np.std(auc_list[valid_auc])  if valid_auc.any() else np.nan

            # ===== STAMPA =====
            print(f"\nTarget: {target_name}")
            print(f"  F1-score     = {f1_mean:.3f}  ±  {f1_std:.3f}")
            print(f"  Accuracy     = {acc_mean:.3f}  ±  {acc_std:.3f}")
            print(
                f"  AUC          = {auc_mean:.3f}  ±  {auc_std:.3f}"
                if not np.isnan(auc_mean)
                else f"  AUC          = NaN     ±  NaN"
            )

            # ===== CSV =====
            rows.append({
                "dataset": dataset_name,
                "target": target_name,
                "F1-score": f"{f1_mean:.3f} ± {f1_std:.3f}",
                "Accuracy": f"{acc_mean:.3f} ± {acc_std:.3f}",
                "AUC": (
                    f"{auc_mean:.3f} ± {auc_std:.3f}"
                    if not np.isnan(auc_mean)
                    else "NaN ± NaN"
                )
            })


    # ===== SALVATAGGIO FILE =====
    if save_csv and rows:
        df_out = pd.DataFrame(rows)
        output_path = Path(output_path)
        df_out.to_csv(output_path, index=False)
        print(f"\n Risultati salvati in: {output_path.resolve()}")

# Lettura dei file

In [6]:
start_time = time.time()

# Eseguo il training per tutti i dataset
results_per_dataset = {}

for name, file_path in datasets.items():

    name_lower = name.lower()

    if "ambl" in name_lower:
        print(f"\n>>> Training AMBL: {name}")
        results_per_dataset[name] = training_ambl(file_path, name)

    elif "duke" in name_lower:
        print(f"\n>>> Training DUKE: {name}")
        results_per_dataset[name] = training_duke(file_path, name)

    else:
        raise ValueError(f"Dataset non riconosciuto: {name}")
    
# Stampa risultati
print_grid_search_results(results_per_dataset)

end_time = time.time()

# Tempo totale
execution_time = end_time - start_time
formatted_time = str(timedelta(seconds=int(execution_time)))

print("\n" + "=" * 80)
print(f" TEMPO TOTALE DI ESECUZIONE: {formatted_time}")
print("=" * 80 + "\n")



>>> Training DUKE: duke_lesions_radiomic

>>> Training DUKE: duke_lesions

>>> Training AMBL: ambl_lesions_radiomic
Fitting 2 folds for each of 27 candidates, totalling 54 fits

[DEBUG] Fold 0
  Target: ER_class | y_true classes: [0 1 2 3] | proba shape: (30, 4)
  Target: PR_class | y_true classes: [0 1 2 3] | proba shape: (30, 4)
  Target: HER2_class | y_true classes: [0 1 3] | proba shape: (30, 3)

Metriche Fold 0
  ER_class: F1=0.479 | ACC=0.700 | AUC=0.601
  PR_class: F1=0.565 | ACC=0.800 | AUC=0.885
  HER2_class: F1=0.430 | ACC=0.700 | AUC=0.698

[DEBUG] Fold 1
  Target: ER_class | y_true classes: [0 1 2 3] | proba shape: (33, 4)
  Target: PR_class | y_true classes: [0 1 2 3] | proba shape: (33, 4)
  Target: HER2_class | y_true classes: [0 1 3] | proba shape: (33, 3)

Metriche Fold 1
  ER_class: F1=0.400 | ACC=0.697 | AUC=0.639
  PR_class: F1=0.465 | ACC=0.667 | AUC=0.906
  HER2_class: F1=0.582 | ACC=0.909 | AUC=0.876

>>> Training AMBL: ambl_lesions
Fitting 2 folds for each of 2

# Validazione Multicentrica

In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

def validazione_multicentrica( df_train, df_test, feature_cols, target_col, n_splits=5, random_state=42):

    X_train = df_train[feature_cols].copy()
    y_train = df_train[target_col].astype(int).copy()

    X_test  = df_test[feature_cols].copy()
    y_test  = df_test[target_col].astype(int).copy()

    pipe = Pipeline([
        ("clf", RandomForestClassifier(
            n_estimators=300,
            random_state=random_state,
            n_jobs=-1,
            class_weight="balanced"
        ))
    ])

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    cv_auc = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        pipe.fit(X_tr, y_tr)
        y_val_prob = pipe.predict_proba(X_val)[:, 1]
        cv_auc.append(roc_auc_score(y_val, y_val_prob))

    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    return {
        "cv_auc_mean": np.mean(cv_auc),
        "cv_auc_std":  np.std(cv_auc),
        "test_auc":    roc_auc_score(y_test, y_prob),
        "test_f1":     f1_score(y_test, y_pred),
        "test_acc":    accuracy_score(y_test, y_pred),
        "n_train":     len(df_train),
        "n_test":      len(df_test),
        "model":       pipe
    }


In [8]:
def create_targets_ambl(df):
    df = df.copy()

    df["ER_class"] = (
        pd.to_numeric(df["ER [SII]"], errors="coerce") >= 1
    ).astype(int)

    df["PR_class"] = (
        pd.to_numeric(df["PR [SII]"], errors="coerce") >= 1
    ).astype(int)

    df["HER2_class"] = (
        pd.to_numeric(df["HER2 [SII]"], errors="coerce") >= 3
    ).astype(int)

    return df


def create_targets_duke(df):

    df["ER_class"]   = pd.to_numeric(df["ER"], errors="coerce")
    df["PR_class"]   = pd.to_numeric(df["PR"], errors="coerce")
    df["HER2_class"] = pd.to_numeric(df["HER2"], errors="coerce")

    return df

In [9]:
"""
    AMBL = training (con CV interna)
    DUKE = test esterno indipendente
"""

# Carico i csv di prova
ambl = pd.read_csv("/Users/francesco/Tesi/BC-ML4/dataset/cleaned/ambl_lesions.csv")
duke = pd.read_csv("/Users/francesco/Tesi/BC-ML4/dataset/cleaned/duke_lesions.csv")

# Creo tutti i target
ambl = create_targets_ambl(ambl)
duke = create_targets_duke(duke)

# Filtro le feature che mi servono
feature_cols = sorted(
    set(ambl.columns)
    .intersection(set(duke.columns))
    - {
        "Patient ID", "lesion idx",
        "ER [SII]", "PR [SII]", "HER2 [SII]",
        "ER_class", "PR_class", "HER2_class"
    }
)
# Validazione multicentrica
targets = ["ER_class", "PR_class", "HER2_class"]
all_results = {}
for target in targets:

    ambl_t = ambl.dropna(subset=[target])
    duke_t = duke.dropna(subset=[target])

    if ambl_t[target].nunique() < 2 or duke_t[target].nunique() < 2:
        print(f"[SKIP] {target} – una sola classe")
        continue

    results = validazione_multicentrica(
        df_train=ambl_t,
        df_test=duke_t,
        feature_cols=feature_cols,
        target_col=target
    )

    all_results[target] = results


In [10]:
rows = []

for target, res in all_results.items():
    rows.append({
        "Target": target.replace("_class", ""),
        "CV AUC (AMBL)": f"{res['cv_auc_mean']:.3f} ± {res['cv_auc_std']:.3f}",
        "Duke AUC": f"{res['test_auc']:.3f}",
        "Duke F1":  f"{res['test_f1']:.3f}",
        "Duke ACC": f"{res['test_acc']:.3f}",
        "N Train":  res["n_train"],
        "N Test":   res["n_test"],
    })

df = pd.DataFrame(rows)

# Ordine colonne (esplicito)
cols = [
    "Target",
    "CV AUC (AMBL)",
    "Duke AUC",
    "Duke F1",
    "Duke ACC",
    "N Train",
    "N Test",
]

df = df[cols]

# Larghezza colonne
col_widths = {
    "Target": 6,
    "CV AUC (AMBL)": 16,
    "Duke AUC": 9,
    "Duke F1": 8,
    "Duke ACC": 9,
    "N Train": 9,
    "N Test": 8,
}

def format_row(row):
    return " | ".join(
        f"{str(row[c]):<{col_widths[c]}}" for c in cols
    )

# Header
header = " | ".join(f"{c:<{col_widths[c]}}" for c in cols)
separator = "-+-".join("-" * col_widths[c] for c in cols)

print("\n=== RISULTATI DELLA VALIDAZIONE MULTICENTRICA ===\n")
print(header)
print(separator)
for _, r in df.iterrows():
    print(format_row(r))



=== RISULTATI DELLA VALIDAZIONE MULTICENTRICA ===

Target | CV AUC (AMBL)    | Duke AUC  | Duke F1  | Duke ACC  | N Train   | N Test  
-------+------------------+-----------+----------+-----------+-----------+---------
ER     | 0.650 ± 0.133    | 0.469     | 0.433    | 0.450     | 82        | 291     
PR     | 0.641 ± 0.100    | 0.488     | 0.332    | 0.529     | 82        | 291     
HER2   | 0.646 ± 0.231    | 0.520     | 0.023    | 0.708     | 82        | 291     


# Stampo la CV

In [11]:
pd.set_option("display.max_colwidth", None)

for name, res in results_per_dataset.items():
    print(f"\n### {name}")

    df = pd.DataFrame(res["cv_results"])

    display(
        df.sort_values("rank_test_score")[
            ["params", "mean_test_score", "std_test_score", "rank_test_score"]
        ].head(10)
    )



### duke_lesions_radiomic


,params,mean_test_score,std_test_score,rank_test_score
2,"{'estimator__max_depth': 2, 'estimator__min_samples_leaf': 1, 'estimator__n_estimators': 100}",0.501411,0.031690,1
5,"{'estimator__max_depth': 2, 'estimator__min_samples_leaf': 2, 'estimator__n_estimators': 100}",0.500447,0.036990,2
8,"{'estimator__max_depth': 2, 'estimator__min_samples_leaf': 3, 'estimator__n_estimators': 100}",0.494583,0.033368,3
0,"{'estimator__max_depth': 2, 'estimator__min_samples_leaf': 1, 'estimator__n_estimators': 50}",0.493747,0.017255,4
13,"{'estimator__max_depth': 4, 'estimator__min_samples_leaf': 2, 'estimator__n_estimators': 75}",0.492923,0.039245,5
26,"{'estimator__max_depth': 6, 'estimator__min_samples_leaf': 3, 'estimator__n_estimators': 100}",0.491149,0.030188,6
3,"{'estimator__max_depth': 2, 'estimator__min_samples_leaf': 2, 'estimator__n_estimators': 50}",0.490743,0.021767,7
4,"{'estimator__max_depth': 2, 'estimator__min_samples_leaf': 2, 'estimator__n_estimators': 75}",0.490475,0.027453,8
1,"{'estimator__max_depth': 2, 'estimator__min_samples_leaf': 1, 'estimator__n_estimators': 75}",0.487178,0.029304,9
16,"{'estimator__max_depth': 4, 'estimator__min_samples_leaf': 3, 'estimator__n_estimators': 75}",0.485649,0.048631,10



### duke_lesions


,params,mean_test_score,std_test_score,rank_test_score
19,"{'estimator__max_depth': 6, 'estimator__min_samples_leaf': 1, 'estimator__n_estimators': 75}",0.506250,0.040430,1
6,"{'estimator__max_depth': 2, 'estimator__min_samples_leaf': 3, 'estimator__n_estimators': 50}",0.506138,0.040527,2
17,"{'estimator__max_depth': 4, 'estimator__min_samples_leaf': 3, 'estimator__n_estimators': 100}",0.505175,0.037325,3
3,"{'estimator__max_depth': 2, 'estimator__min_samples_leaf': 2, 'estimator__n_estimators': 50}",0.504865,0.043312,4
0,"{'estimator__max_depth': 2, 'estimator__min_samples_leaf': 1, 'estimator__n_estimators': 50}",0.502680,0.042775,5
7,"{'estimator__max_depth': 2, 'estimator__min_samples_leaf': 3, 'estimator__n_estimators': 75}",0.502576,0.034792,6
2,"{'estimator__max_depth': 2, 'estimator__min_samples_leaf': 1, 'estimator__n_estimators': 100}",0.502287,0.040948,7
23,"{'estimator__max_depth': 6, 'estimator__min_samples_leaf': 2, 'estimator__n_estimators': 100}",0.501116,0.041093,8
18,"{'estimator__max_depth': 6, 'estimator__min_samples_leaf': 1, 'estimator__n_estimators': 50}",0.499632,0.027093,9
20,"{'estimator__max_depth': 6, 'estimator__min_samples_leaf': 1, 'estimator__n_estimators': 100}",0.499457,0.034322,10



### ambl_lesions_radiomic


,params,mean_test_score,std_test_score,rank_test_score
4,"{'estimator__max_depth': 2, 'estimator__min_samples_leaf': 2, 'estimator__n_estimators': 75}",0.486751,0.004638,1
14,"{'estimator__max_depth': 4, 'estimator__min_samples_leaf': 2, 'estimator__n_estimators': 100}",0.486618,0.010581,2
22,"{'estimator__max_depth': 6, 'estimator__min_samples_leaf': 2, 'estimator__n_estimators': 75}",0.486538,0.006576,3
7,"{'estimator__max_depth': 2, 'estimator__min_samples_leaf': 3, 'estimator__n_estimators': 75}",0.484423,0.015731,4
5,"{'estimator__max_depth': 2, 'estimator__min_samples_leaf': 2, 'estimator__n_estimators': 100}",0.480712,0.000967,5
3,"{'estimator__max_depth': 2, 'estimator__min_samples_leaf': 2, 'estimator__n_estimators': 50}",0.478700,0.005124,6
16,"{'estimator__max_depth': 4, 'estimator__min_samples_leaf': 3, 'estimator__n_estimators': 75}",0.474959,0.021429,7
25,"{'estimator__max_depth': 6, 'estimator__min_samples_leaf': 3, 'estimator__n_estimators': 75}",0.474959,0.021429,7
8,"{'estimator__max_depth': 2, 'estimator__min_samples_leaf': 3, 'estimator__n_estimators': 100}",0.473731,0.004282,9
1,"{'estimator__max_depth': 2, 'estimator__min_samples_leaf': 1, 'estimator__n_estimators': 75}",0.472465,0.011384,10



### ambl_lesions


,params,mean_test_score,std_test_score,rank_test_score
24,"{'estimator__max_depth': 6, 'estimator__min_samples_leaf': 3, 'estimator__n_estimators': 50}",0.483025,0.017144,1
15,"{'estimator__max_depth': 4, 'estimator__min_samples_leaf': 3, 'estimator__n_estimators': 50}",0.483025,0.017144,1
6,"{'estimator__max_depth': 2, 'estimator__min_samples_leaf': 3, 'estimator__n_estimators': 50}",0.465150,0.028222,3
3,"{'estimator__max_depth': 2, 'estimator__min_samples_leaf': 2, 'estimator__n_estimators': 50}",0.462555,0.010553,4
26,"{'estimator__max_depth': 6, 'estimator__min_samples_leaf': 3, 'estimator__n_estimators': 100}",0.459268,0.016543,5
17,"{'estimator__max_depth': 4, 'estimator__min_samples_leaf': 3, 'estimator__n_estimators': 100}",0.459268,0.016543,5
16,"{'estimator__max_depth': 4, 'estimator__min_samples_leaf': 3, 'estimator__n_estimators': 75}",0.455205,0.020993,7
25,"{'estimator__max_depth': 6, 'estimator__min_samples_leaf': 3, 'estimator__n_estimators': 75}",0.455205,0.020993,7
7,"{'estimator__max_depth': 2, 'estimator__min_samples_leaf': 3, 'estimator__n_estimators': 75}",0.455068,0.011166,9
5,"{'estimator__max_depth': 2, 'estimator__min_samples_leaf': 2, 'estimator__n_estimators': 100}",0.452128,0.021802,10
